# BanglaBERT Error Analysis — Subgroup Breakdown

**Authors:** Swagotam Malakar, Anamika Das, Dr. Ohidujjaman
**Environment:** Kaggle CPU (no GPU required if predictions CSV is supplied); optional GPU fallback path for re-running NB1's CV
**Estimated runtime:** ~5 minutes (CPU, if `banglabert_clean_predictions.csv` is supplied as input) — OR ~2 hours (GPU, if NB1's 5-fold CV must be re-run from scratch)
**Last validated:** 2026-07-21

---

## Purpose

This notebook analyzes **BanglaBERT Large**'s misclassifications on the
**766-article gold standard** (`Swarabyanjan_Gold_Balanced_766.csv` /
`Swarabyanjan_BEST_BALANCED_1to1.csv`).

BanglaBERT Large achieves F1 = 0.883 (5-fold CV) on these 766 articles —
the best-performing model in the Swarabyanjan benchmark. Aggregate F1 alone
does not characterise the model's failure modes; we therefore ask:

> *"What does BanglaBERT get wrong? Are the errors systematic?"*

We break down errors by **six dimensions**:

1. **Annotation confidence** (H/M/L) — does BanglaBERT struggle on
   articles that human annotators themselves found ambiguous?
2. **Corpus batch** (7 batch tags) — is performance uneven across
   the corpus-expansion vs the stratified-resample batches?
3. **Article length** (5 bins: 0-500, 500-1K, 1K-2K, 2K-4K, 4K+ chars)
   — does BanglaBERT degrade on very short or very long articles?
4. **True label** — false positives (predicted yellow, actually
   non-yellow) vs false negatives (predicted non-yellow, actually
   yellow), with 5 example headlines + annotator notes each.
5. **SMI criteria scores C1-C8** — for each criterion, t-test whether
   BanglaBERT errors correlate with the criterion score.
6. **Probability calibration** — Brier score + Expected Calibration
   Error (ECE) + reliability diagram. Are BanglaBERT's probability
   estimates well-calibrated?

## Reproducibility

- Random seed: `SEED = 42` (matches NB1's CV seed).
- Gold standard CSV: `Swarabyanjan_Gold_Balanced_766.csv` (cleaned) preferred,
  `Swarabyanjan_BEST_BALANCED_1to1.csv` (legacy) supported as fallback.
- BanglaBERT predictions: **prefer loading a saved CSV
  (`banglabert_clean_predictions.csv`)** produced by NB1. If absent, the
  notebook can optionally re-run NB1's 5-fold CV inline (`RUN_CV_FALLBACK = True`)
  — this requires GPU + `torch` + `transformers`.
- All SMI scoring functions are copied **verbatim** from NB8 to ensure
  identical C1-C8 scores across notebooks.


## Kaggle Setup

| Setting | Value |
|---------|-------|
| Accelerator | **None (CPU only)** — default; or **GPU T4** if re-running BanglaBERT CV |
| Internet | Off (default); On (if GPU fallback enabled) |
| Expected runtime | ~5 min (CPU, with saved predictions) or ~2 hours (GPU fallback) |

**Required Kaggle Inputs:**
- Dataset: `swagotammalakar/v18-human-gold-final` (provides gold CSV)
- Optional: Dataset with `banglabert_clean_predictions.csv` (766 rows, NB1 gold output) OR `banglabert_all_predictions_5000.csv` (5000 rows, full corpus) — enables CPU mode. The 5000-row file is filtered to the 766 gold articles by `article_id`/`v18_id` at load time.

> **Note:** If Kaggle resets the inputs/accelerator after re-importing, re-attach the dataset(s) listed above. Default is CPU-only; only enable GPU if setting `RUN_CV_FALLBACK = True` in the configuration cell.


### 1. Environment Setup

Imports only — pandas, numpy, sklearn, matplotlib, seaborn, scipy.stats.
No `torch`, no `transformers` is imported at the top level; the GPU
fallback path imports them lazily inside the conditional that needs them.

This is a **CPU-only** notebook when the saved-predictions path is used.


In [1]:
# === SETUP ===
import os
import sys
import json
import glob
import time
import warnings
import random
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')  # headless backend (Kaggle commit mode)
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    cohen_kappa_score, matthews_corrcoef, roc_auc_score,
    confusion_matrix, classification_report,
)
from scipy import stats

warnings.filterwarnings('ignore')

# Font that handles basic characters and minus signs
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"pandas {pd.__version__}, numpy {np.__version__}", flush=True)
print(f"matplotlib {matplotlib.__version__}, seaborn {sns.__version__}", flush=True)
print(f"scipy {stats.__version__ if hasattr(stats, '__version__') else 'n/a'}", flush=True)
print(f"NB14: BanglaBERT error analysis (CPU-only main path)", flush=True)


pandas 2.3.3, numpy 2.0.2
matplotlib 3.10.0, seaborn 0.13.2
scipy n/a
NB14: BanglaBERT error analysis (CPU-only main path)


### 2. Configuration — File Paths & Flags

- `GOLD_FILENAME`: cleaned gold CSV (preferred) → falls back to legacy CSV.
- `PRED_FILENAME`: NB1's saved predictions CSV. **If found**, the notebook
  runs in ~5 minutes on CPU. If not found, set `RUN_CV_FALLBACK = True`
  to re-run NB1's 5-fold CV inline (requires GPU + `torch` + `transformers`).
- `OUTPUT_DIR`: Kaggle working dir if available, else local `./nb14_outputs/`.


In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

# --- Gold standard CSV ---
# Prefer the cleaned CSV (Task 8 output); fall back to the legacy CSV.
GOLD_FILENAME_PRIMARY = 'Swarabyanjan_Gold_Balanced_766.csv'
GOLD_FILENAME_LEGACY  = 'Swarabyanjan_BEST_BALANCED_1to1.csv'

# --- BanglaBERT predictions CSV (saved by NB1 cell 9) ---
PRED_FILENAME = 'banglabert_clean_predictions.csv'
# Also try these alternate names produced by various NB1 runs
PRED_ALT_NAMES = [
    'banglabert_clean_predictions.csv',        # 766-row gold-only predictions (NB1 output)
    'banglabert_predictions.csv',
    'nb1_banglabert_predictions.csv',
    'banglabert_all_predictions_5000.csv',     # 5000-row full-corpus predictions (filtered to 766 below)
]

# --- CV fallback flag ---
# If True and no saved predictions CSV is found, re-run NB1's 5-fold CV
# inline. REQUIRES GPU + torch + transformers + ~2 hours.
# Default False keeps the notebook CPU-only — the user must set this to
# True explicitly to enable the GPU re-run path.
RUN_CV_FALLBACK = False

# --- Output directory ---
OUTPUT_DIR = Path('/kaggle/working') if os.path.exists('/kaggle/working') else Path('./nb14_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# BanglaBERT hyperparameters (used ONLY if CV fallback is triggered)
MODEL_NAME      = 'csebuetnlp/banglabert_large'
MAX_LEN         = 512
BATCH_SIZE      = 16
EPOCHS          = 4
LEARNING_RATE   = 2e-5
WARMUP_RATIO    = 0.1
WEIGHT_DECAY    = 0.01
N_FOLDS         = 5


def find_file(filenames):
    """Search Kaggle input paths + local fallbacks for any of the filenames.

    `filenames` is a list of candidate filenames tried in order.
    Returns the first match found, or None if nothing matches.
    """
    if isinstance(filenames, str):
        filenames = [filenames]
    for fname in filenames:
        # Kaggle input — recursive search
        for pattern in [
            f'/kaggle/input/**/{fname}',
            f'/kaggle/working/{fname}',
        ]:
            matches = glob.glob(pattern, recursive=True)
            if matches:
                return matches[0]
        # Local fallbacks (for development / testing)
        for local in [
            f'./{fname}',
            f'./data/{fname}',
            f'./results/{fname}',
            f'../data/{fname}',
            f'../results/{fname}',
            f'/home/z/my-project/analysis/github_repo/data/{fname}',
            f'/home/z/my-project/analysis/github_repo/results/{fname}',
            f'/home/z/my-project/download/{fname}',
            f'/home/z/my-project/upload/{fname}',
        ]:
            if os.path.isfile(local):
                return local
    return None


GOLD_PATH = find_file([GOLD_FILENAME_PRIMARY, GOLD_FILENAME_LEGACY])
PRED_PATH = find_file(PRED_ALT_NAMES)

print(f"Gold CSV:                {GOLD_PATH}", flush=True)
print(f"BanglaBERT preds CSV:    {PRED_PATH}", flush=True)
print(f"Output dir:              {OUTPUT_DIR}", flush=True)
print(f"RUN_CV_FALLBACK flag:    {RUN_CV_FALLBACK}", flush=True)

if GOLD_PATH is None:
    raise FileNotFoundError(
        f'Could not find gold CSV. Looked for {GOLD_FILENAME_PRIMARY} '
        f'(primary) and {GOLD_FILENAME_LEGACY} (legacy) under '
        f'/kaggle/input/**, ./, ../data/, and local repo paths. '
        f'Upload the gold standard CSV as a Kaggle dataset and attach it '
        f'as input to this notebook.'
    )


Gold CSV:                /kaggle/input/datasets/smalakarishere/swarabyanjan/Swarabyanjan_Gold_Balanced_766.csv
BanglaBERT preds CSV:    /kaggle/input/notebooks/swagotammalakar/nb1-banglabert-classical/banglabert_clean_predictions.csv
Output dir:              /kaggle/working
RUN_CV_FALLBACK flag:    False


### 3. Load Gold Standard

Load the 766-article gold standard with all metadata:
`article_id, headline, body_text, corpus_batch, article_length,
best_label, best_confidence, best_note`.

Two schemas are supported:
- **Cleaned** (`Swarabyanjan_Gold_Balanced_766.csv`): has `corpus_batch`.
- **Legacy** (`Swarabyanjan_BEST_BALANCED_1to1.csv`): has `news_source`
  (which actually contains the corpus_batch tag). We alias it to
  `corpus_batch` for downstream code.


In [3]:
# === LOAD GOLD STANDARD ===
df = pd.read_csv(GOLD_PATH)
print(f'Gold loaded: {df.shape}', flush=True)
print(f'Columns: {list(df.columns)}', flush=True)

# --- Column auto-detection (matches NB8's detect_gold_columns logic) ---
# The cleaned CSV has corpus_batch; the legacy CSV has news_source.
if 'corpus_batch' in df.columns:
    BATCH_COL = 'corpus_batch'
elif 'news_source' in df.columns:
    BATCH_COL = 'news_source'
    df = df.rename(columns={'news_source': 'corpus_batch'})
    print('  (renamed news_source -> corpus_batch for downstream consistency)', flush=True)
else:
    # corpus_batch unknown — fill with 'unknown' so crosstabs still work
    df['corpus_batch'] = 'unknown'
    BATCH_COL = 'corpus_batch'
    print('  WARNING: no corpus_batch/news_source column found; using "unknown".', flush=True)

# Ensure text columns are strings (handle NaN)
df['headline']    = df['headline'].fillna('').astype(str)
df['body_text']   = df['body_text'].fillna('').astype(str)
df['best_note']   = df['best_note'].fillna('').astype(str)
df['best_label']  = df['best_label'].astype(int)

# Handle stub articles (body_text == "not_available") — same as NB8
n_stub = int((df['body_text'] == 'not_available').sum())
if n_stub > 0:
    print(f'  {n_stub} stub articles (body_text="not_available") — '
          f'replaced with empty string for SMI scoring.', flush=True)
    df.loc[df['body_text'] == 'not_available', 'body_text'] = ''

# Alias for convenience
y_true = df['best_label'].values
n_total = len(df)

print(f'\nGold standard: {n_total} articles', flush=True)
print(f'  Yellow (1):     {int(y_true.sum())}', flush=True)
print(f'  Non-yellow (0): {int((y_true == 0).sum())}', flush=True)
print(f'\nbest_confidence distribution:', flush=True)
print(df['best_confidence'].value_counts().to_string(), flush=True)
print(f'\ncorpus_batch distribution:', flush=True)
print(df['corpus_batch'].value_counts().to_string(), flush=True)
print(f'\narticle_length stats:', flush=True)
print(df['article_length'].describe().round(0).to_string(), flush=True)
df.head(3)


Gold loaded: (766, 8)
Columns: ['article_id', 'headline', 'body_text', 'corpus_batch', 'article_length', 'best_label', 'best_confidence', 'best_note']
  2 stub articles (body_text="not_available") — replaced with empty string for SMI scoring.

Gold standard: 766 articles
  Yellow (1):     383
  Non-yellow (0): 383

best_confidence distribution:
best_confidence
H    376
M    328
L     62

corpus_batch distribution:
corpus_batch
corpus_expansion_5000    649
v17_reused                48
new_low_w0_pany           19
new_mid_w0_p0             16
new_mid_w1_p1             14
new_mid_w1_p0             10
new_high_w1_pany          10

article_length stats:
count     766.0
mean     1615.0
std      1194.0
min        13.0
25%       905.0
50%      1231.0
75%      1944.0
max      8365.0


,article_id,headline,body_text,corpus_batch,article_length,best_label,best_confidence,best_note
0,v18_2830,কুমিল্লায় ‘ডাকাতের গুলিতে ডাকাত’ নিহত,"দাউদকান্দিথানার ওসি মিজানুর রহমান বলেন, “দুই দ...",corpus_expansion_5000,694,0,H,Crime news with named Daudkandi OC Mizanur Rah...
1,v18_0000,২০১৮ সালে বিশ্বজুড়ে ৯৭ সাংবাদিক খুন,বিশ্বজুড়ে সাংবাদিকদের ওপর হামলার ঘটনাগুলোতে ব...,new_low_w0_pany,1205,0,H,Factual international press-freedom report wit...
2,v18_4780,বিশ্বকাপে আফগানদের বিপক্ষে টেস্ট খেললেন ধোনি?,খেলা শুরুর আগেই 'ফেভারিট বনাম লাস্টবয়' শিরোনা...,corpus_expansion_5000,967,1,M,[R1-BROADER] Clickbait sports headline ('?'); ...


### 4. Load or Regenerate BanglaBERT Predictions

We need BanglaBERT's cross-validated predictions on the 766 articles:
- `banglabert_all_preds` — 766 integer class predictions (0/1)
- `banglabert_all_probs` — 766 float probabilities of class 1

**Primary path (CPU only, ~5 min):** load the saved predictions CSV
produced by NB1's cell 9 (`banglabert_clean_predictions.csv`). The CSV
has columns `article_id, true_label, banglabert_pred, banglabert_prob`.

**Fallback path (GPU, ~2 hours):** if no predictions CSV is found AND
`RUN_CV_FALLBACK = True`, re-run NB1's 5-fold CV inline. This requires
`torch`, `transformers`, and a GPU. **The default is False** — the user
must explicitly opt in by flipping the flag in the Configuration cell.

We use the same 5-fold CV predictions from NB1 (seed=42).


In [4]:
# === LOAD OR REGENERATE BANGLABERT PREDICTIONS ===

banglabert_all_preds = None
banglabert_all_probs = None
PRED_SOURCE = None  # 'csv' or 'cv_fallback'

if PRED_PATH is not None:
    print(f'[Branch A] Loading saved BanglaBERT predictions from: {PRED_PATH}', flush=True)
    pred_df = pd.read_csv(PRED_PATH)
    print(f'  Predictions CSV shape: {pred_df.shape}', flush=True)
    print(f'  Columns: {list(pred_df.columns)}', flush=True)

    # Detect prediction / probability / id columns
    pred_col = None
    for c in pred_df.columns:
        if c.lower() in ('banglabert_pred', 'pred', 'prediction', 'y_pred'):
            pred_col = c; break
    prob_col = None
    for c in pred_df.columns:
        if c.lower() in ('banglabert_prob', 'prob', 'probability', 'y_prob', 'prob_1'):
            prob_col = c; break
    id_col = None
    for c in pred_df.columns:
        # Accept 'article_id', 'id', and 'v18_id' (the latter is the column
        # name used by the full-corpus 5000-row predictions CSV).
        if c.lower() in ('article_id', 'id', 'v18_id'):
            id_col = c; break

    if pred_col is None:
        raise ValueError(
            f'Predictions CSV has no prediction column. '
            f'Expected one of banglabert_pred/pred/prediction/y_pred. '
            f'Found: {list(pred_df.columns)}'
        )
    if prob_col is None:
        raise ValueError(
            f'Predictions CSV has no probability column. '
            f'Expected one of banglabert_prob/prob/probability/y_prob. '
            f'Found: {list(pred_df.columns)}'
        )

    # Align predictions to gold standard by article_id (or v18_id).
    # This handles BOTH the 766-row gold-only CSV AND the 5000-row
    # full-corpus CSV (filtering down to the 766 gold article_ids).
    if id_col is not None and 'article_id' in df.columns:
        # Rename the id/pred/prob columns to canonical names. If the CSV
        # uses 'v18_id', this renames it to 'article_id' for alignment.
        pred_df = pred_df.rename(columns={id_col: 'article_id',
                                          pred_col: 'banglabert_pred',
                                          prob_col: 'banglabert_prob'})
        n_before = len(pred_df)

        # Filter to the gold article_ids (drops the 4234 non-gold rows
        # when the CSV is the full 5000-row corpus; no-op when the CSV
        # is already 766 rows).
        gold_ids = set(df['article_id'].astype(str))
        pred_df = pred_df[pred_df['article_id'].astype(str).isin(gold_ids)].copy()
        n_after = len(pred_df)
        if n_before != n_after:
            print(f'  Filtered predictions: {n_before} -> {n_after} rows '
                  f'(kept only gold article_ids)', flush=True)

        # Drop any duplicate article_ids (keep first occurrence) so the
        # merge is highly stable.
        pred_df = pred_df.drop_duplicates(subset=['article_id'], keep='first')

        # Sort both DataFrames by article_id so the merge preserves the
        # canonical 766-article row order (matches NB1 / NB8 / NB13).
        gold_sorted = df[['article_id']].sort_values('article_id').reset_index(drop=True)
        pred_sorted = pred_df[['article_id', 'banglabert_pred', 'banglabert_prob']].sort_values('article_id').reset_index(drop=True)
        merged = gold_sorted.merge(pred_sorted, on='article_id', how='left')

        if merged['banglabert_pred'].isna().any():
            n_missing = int(merged['banglabert_pred'].isna().sum())
            raise ValueError(
                f'{n_missing} of {len(df)} gold article_ids have no '
                f'prediction in the CSV. Ensure the CSV covers all 766 '
                f'gold articles (or use the CV fallback).'
            )

        # Re-order merged to match the original gold row order (so
        # banglabert_all_preds[i] corresponds to df.iloc[i]).
        merged = df[['article_id']].merge(merged, on='article_id', how='left')
        banglabert_all_preds = merged['banglabert_pred'].astype(int).values
        banglabert_all_probs = merged['banglabert_prob'].astype(float).values
    else:
        # No article_id/v18_id column in the predictions CSV.
        # Require an exact 766-row CSV (already aligned by row order).
        if len(pred_df) != n_total:
            raise ValueError(
                f'Predictions CSV has {len(pred_df)} rows but gold has '
                f'{n_total}. Cannot align without an article_id (or '
                f'v18_id) column. Either (a) ensure the CSV has an '
                f'article_id or v18_id column, or (b) supply a CSV with '
                f'exactly {n_total} rows in the same order as the gold.'
            )
        banglabert_all_preds = pred_df[pred_col].astype(int).values
        banglabert_all_probs = pred_df[prob_col].astype(float).values

    PRED_SOURCE = 'csv'
    print(f'  Loaded {len(banglabert_all_preds)} predictions.', flush=True)
    print(f'  Predicted positives (1): {int(banglabert_all_preds.sum())}', flush=True)
    print(f'  Predicted negatives (0): {int((banglabert_all_preds == 0).sum())}', flush=True)

elif RUN_CV_FALLBACK:
    print('[Branch B] RE-RUNNING NB1\'s 5-fold CV (GPU required, ~2 hours)...', flush=True)
    print('  Importing torch + transformers (lazy import)...', flush=True)

    # Lazy imports — only loaded here so the rest of the notebook stays CPU-only.
    import torch  # noqa: E402
    from torch.utils.data import Dataset  # noqa: E402
    from transformers import (  # noqa: E402
        AutoTokenizer, AutoModelForSequenceClassification,
        Trainer, TrainingArguments, DataCollatorWithPadding,
    )

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'  Device: {DEVICE}', flush=True)
    if DEVICE.type == 'cpu':
        raise RuntimeError(
            'RUN_CV_FALLBACK=True but no GPU detected. '
            'BanglaBERT Large 5-fold CV on CPU would take days. '
            'Either (a) attach a GPU to this notebook, or '
            '(b) supply a saved banglabert_clean_predictions.csv '
            'as input and keep RUN_CV_FALLBACK=False.'
        )
    if torch.cuda.is_available():
        print(f'  GPU: {torch.cuda.get_device_name(0)}', flush=True)
        print(f'  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB', flush=True)

    # --- Tokenizer + dataset (copied from NB1) ---
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    print(f'  Tokenizer loaded: {MODEL_NAME}', flush=True)

    class BengaliNewsDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
            self.texts = texts.tolist() if hasattr(texts, 'tolist') else list(texts)
            self.labels = labels.tolist() if hasattr(labels, 'tolist') else list(labels)
            self.tokenizer = tokenizer
            self.max_len = max_len
        def __len__(self):
            return len(self.texts)
        def __getitem__(self, idx):
            enc = self.tokenizer(
                self.texts[idx],
                truncation=True, max_length=self.max_len, padding=False,
            )
            return {
                'input_ids': enc['input_ids'],
                'attention_mask': enc['attention_mask'],
                'labels': int(self.labels[idx]),
            }

    def compute_metrics_for_trainer(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
        return {
            'accuracy': accuracy_score(labels, preds),
            'f1': f1_score(labels, preds, zero_division=0),
            'kappa': cohen_kappa_score(labels, preds),
            'mcc': matthews_corrcoef(labels, preds),
            'roc_auc': roc_auc_score(labels, probs) if len(set(labels)) > 1 else 0.0,
        }

    # --- 5-fold CV (mirrors NB1 cell 8) ---
    X_text = (df['headline'] + ' ' + df['body_text']).values
    folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X_text, y_true))

    banglabert_all_preds = np.zeros(len(y_true), dtype=int)
    banglabert_all_probs = np.full(len(y_true), np.nan)
    fold_results = []

    import shutil
    for fold_i, (train_idx, val_idx) in enumerate(folds):
        print(f'\n{"="*60}\nBanglaBERT Fold {fold_i+1}/{N_FOLDS}  '
              f'(train={len(train_idx)}, val={len(val_idx)})\n{"="*60}', flush=True)

        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
        train_ds = BengaliNewsDataset(X_text[train_idx], y_true[train_idx], tokenizer)
        val_ds   = BengaliNewsDataset(X_text[val_idx],   y_true[val_idx],   tokenizer)
        collator = DataCollatorWithPadding(tokenizer=tokenizer)

        fold_output_dir = OUTPUT_DIR / f'banglabert_fold_{fold_i+1}'
        training_args = TrainingArguments(
            output_dir=str(fold_output_dir),
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE * 2,
            learning_rate=LEARNING_RATE,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            logging_steps=50,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='f1',
            greater_is_better=True,
            save_total_limit=1,
            report_to='none',
            seed=SEED,
            fp16=True,
            gradient_accumulation_steps=1,
            dataloader_num_workers=2,
        )
        trainer = Trainer(
            model=model, args=training_args,
            train_dataset=train_ds, eval_dataset=val_ds,
            data_collator=collator,
            compute_metrics=compute_metrics_for_trainer,
        )
        trainer.train()
        predictions = trainer.predict(val_ds)
        logits = predictions.predictions
        fold_preds = np.argmax(logits, axis=-1)
        fold_probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

        banglabert_all_preds[val_idx] = fold_preds
        banglabert_all_probs[val_idx] = fold_probs

        fm = {
            'fold': fold_i + 1,
            'accuracy':  accuracy_score(y_true[val_idx], fold_preds),
            'f1':        f1_score(y_true[val_idx], fold_preds, zero_division=0),
            'precision': precision_score(y_true[val_idx], fold_preds, zero_division=0),
            'recall':    recall_score(y_true[val_idx], fold_preds, zero_division=0),
            'kappa':     cohen_kappa_score(y_true[val_idx], fold_preds),
            'mcc':       matthews_corrcoef(y_true[val_idx], fold_preds),
            'roc_auc':   roc_auc_score(y_true[val_idx], fold_probs) if len(set(y_true[val_idx])) > 1 else 0.0,
        }
        fold_results.append(fm)
        print(f'  Fold {fold_i+1}: F1={fm["f1"]:.4f}  Acc={fm["accuracy"]:.4f}  '
              f'Kappa={fm["kappa"]:.4f}  MCC={fm["mcc"]:.4f}', flush=True)

        del model, trainer
        torch.cuda.empty_cache()
        if fold_output_dir.exists():
            shutil.rmtree(fold_output_dir)

    # Save predictions so future runs can use Branch A
    pred_df = pd.DataFrame({
        'article_id':     df['article_id'].values,
        'true_label':     y_true,
        'banglabert_pred': banglabert_all_preds,
        'banglabert_prob': banglabert_all_probs,
    })
    saved_csv = OUTPUT_DIR / PRED_FILENAME
    pred_df.to_csv(saved_csv, index=False)
    print(f'\nSaved CV predictions to {saved_csv}', flush=True)
    PRED_SOURCE = 'cv_fallback'

    fold_df = pd.DataFrame(fold_results)
    print(f'\n5-fold CV mean F1: {fold_df["f1"].mean():.4f} ± {fold_df["f1"].std():.4f}', flush=True)

else:
    raise FileNotFoundError(
        f'No saved BanglaBERT predictions found (looked for '
        f'{PRED_ALT_NAMES}). '
        f'Two options:\n'
        f'  (A) Upload banglabert_clean_predictions.csv as a Kaggle '
        f'input dataset (produced by NB1 cell 9). This is the CPU-only '
        f'recommended path.\n'
        f'  (B) Set RUN_CV_FALLBACK = True in the Configuration cell '
        f'above to re-run NB1\'s 5-fold CV inline. This requires GPU + '
        f'torch + transformers and takes ~2 hours on a Kaggle T4.'
    )

print(f'\n[Done] banglabert_all_preds: shape={banglabert_all_preds.shape}, '
      f'dtype={banglabert_all_preds.dtype}', flush=True)
print(f'       banglabert_all_probs: shape={banglabert_all_probs.shape}, '
      f'dtype={banglabert_all_probs.dtype}', flush=True)
print(f'       pred source: {PRED_SOURCE}', flush=True)


[Branch A] Loading saved BanglaBERT predictions from: /kaggle/input/notebooks/swagotammalakar/nb1-banglabert-classical/banglabert_clean_predictions.csv
  Predictions CSV shape: (766, 4)
  Columns: ['article_id', 'true_label', 'banglabert_pred', 'banglabert_prob']
  Loaded 766 predictions.
  Predicted positives (1): 401
  Predicted negatives (0): 365

[Done] banglabert_all_preds: shape=(766,), dtype=int64
       banglabert_all_probs: shape=(766,), dtype=float64
       pred source: csv


### 5. Compute SMI Criteria Scores for All 766 Articles

We copy NB8's SMI criteria scoring functions **verbatim** (C1-C8) so
the criteria scores computed here are bit-for-bit identical to those
used in NB8's evaluation. This lets us correlate BanglaBERT errors
with SMI criteria without re-deriving the lexicons.

The 8 criteria are:

| Criterion | Name | Formula |
|-----------|------|---------|
| C1 | Sensational Headline | lexicon hits + punctuation bonus |
| C2 | Clickbait | phrase hits + listicle + trailing-? bonus |
| C3 | Emotional Arousal | 1 − exp(−D/γ), D=density per 100 words |
| C4 | Attribution Gap | max(1 − λ·n_attr − credits, 0) + short penalty |
| C5 | Speculation-as-Fact | 1 − exp(−D/γ) |
| C6 | Entertainment Displacement | min(hits/α + 0.25·headline_hits, 1) |
| C7 | Headline-Body Coherence | 1 − overlap_ratio if overlap < τ, else 0 |
| C8 | Sensitive Topic | 1 − exp(−D/γ) on combined headline+body |


In [5]:
# === SMI CRITERIA SCORING FUNCTIONS ===
# These implement the mathematical definitions C1-C7 from the paper.

import re
import math
import unicodedata

# --- Lexicons ---

SENSATIONAL_HEADLINE_TERMS = [
    "অবিশ্বাস্য", "অকল্পনীয়", "চমকে", "চাঞ্চল্যকর", "রোমহর্ষক",
    "ভয়ঙ্কর", "নারকীয়", "মর্মান্তিক", "বিভীষিকাময়",
    "চরম", "মহা", "প্রচণ্ড", "কেলেঙ্কারি", "কেলো", "হয়রানি",
    "আলোচিত", "বিতর্কিত", "রহস্যময়", "রহস্য",
    "তবে কি", "তবে কী", "কী ঘটল", "কী হলো",
    "রহস্যের", "রহস্য জট", "জট খুলল", "পর্দা ফাঁক",
    "অবাক", "হতবাক", "স্তব্ধ", "বিস্ময়ে হতবাক",
    "কাঁদছে", "ফাটল", "ছিন্নভিন্ন", "তোলপাড়", "নড়েচড়ে",
    "চাঞ্চল্য", "শিহরণ", "আঁতকে", "কাঁপিয়ে", "কাঁপছে",
]

CLICKBAIT_PHRASES = [
    "তবে কি", "তবে কী", "জানলে অবাক", "যা ঘটল", "যা কেউ বলেনি",
    "ভাবেননি", "অবাক করবে", "চমকে দেওয়া", "অজানা সত্য",
    "এক চমকে", "হয়তো ভাবেননি", "যা দেখলে", "বিশ্বাস করবেন না",
    "নিজের চোখে দেখুন", "ভিডিওতে দেখুন", "ছবিতে দেখুন",
    "পুরো ঘটনা", "পুরো রহস্য", "না জানলে মিস", "অপেক্ষা করুন",
    "রহস্যের জট", "মজার", "মজার তথ্য",
    "যা আপনি জানেন না", "গোপন তথ্য", "আসল সত্য",
    "চমকপ্রদ", "নজরকাড়া", "অভাবনীয়",
    "অবশ্যই দেখুন", "শেয়ার করুন", "ভাইরাল",
    "দেখে নিন", "জেনে নিন", "চিনে নিন",
    "বিস্ময়কর", "অকল্পনীয়", "অবিশ্বাস্য",
]

CLICKBAIT_LISTICLE_RE = re.compile(
    r"(\d+|১|২|৩|৪|৫|৬|৭|৮|৯|১০)\s*(টি|টা|ভাবে|কারণে|টিপস|পদ্ধতি|উপায়)"
)

EMOTIONAL_TERMS = [
    "অশ্রু", "কান্না", "হাহাকার", "বিলাপ", "করুণ", "করুণতা",
    "কান্নায় ভেঙে", "শোকে", "শোকাহত", "বিলাপ করছেন",
    "করুণ আর্তনাদ", "আর্তনাদ", "হাহাকার শুরু",
    "বুক ফেটে", "হৃদয় বিদারণ", "মর্মান্তিক", "নারকীয়",
    "বিভীষিকাময়", "রোমহর্ষক", "কম্পিত", "কাঁপছে",
    "হাহাকারে", "হাহাকার উঠেছে", "রোদন",
    "বিষণ্ণ", "হতাশ", "হতাশা", "নিরাশা",
    "উল্লাসে", "উল্লাসিত", "আনন্দে", "আনন্দঘন",
    "ক্ষোভে", "ক্ষুব্ধ", "রুষ্ট", "ক্ষোভ প্রকাশ",
    "বিক্ষোভ", "ধিক্কার", "নিন্দা", "প্রতিবাদ",
]

ATTRIBUTION_TERMS = [
    "বলেন", "জানিয়েছেন", "জানান", "বলা হয়েছে", "বলেছেন",
    "মতে", "অনুসারে", "সূত্রে", "সূত্র বলছে",
    "নিশ্চিত করেছেন", "নিশ্চিত করা হয়েছে",
    "প্রকাশ করেছেন", "প্রকাশ করেছে",
    "জানিয়েছে", "বলা হয়", "যোগ করেছেন",
    "রইটার্স", "রয়টার্স", "রয়টার", "বিডিনিউজ", "বাসস", "ইউএনবি",
    "এএফপি", "এপি", "ডিপিএ",
    "প্রতিবেদক", "প্রতিনিধি", "নিজস্ব প্রতিবেদক",
    "সংস্থা", "সংস্দা", "সংবাদ সংস্থা",
    "বিবৃতি", "প্রেস বিবৃতি", "বিজ্ঞপ্তি", "প্রেস রিলিজ",
    "আদালত", "পুলিশ", "মন্ত্রণালয়", "সরকার", "সংসদ",
    "বিভাগ", "অধিদপ্তর", "পরিষদ", "কমিটি", "কমিশন",
    "টিআইবি", "ট্রান্সপারেন্সি ইন্টারন্যাশনাল",
    "রিপোর্ট", "প্রতিবেদন", "তদন্ত", "অনুসন্ধান",
    "বিশেষজ্ঞ", "বিশ্লেষক", "অধ্যাপক", "ডাক্তার",
    "মামলা", "রায়", "আদেশ", "নোটিশ",
]

SPECULATION_TERMS = [
    "হতে পারে", "হতে পারেন", "থাকতে পারে", "হয়তো", "সম্ভবত",
    "মনে হচ্ছে", "মনে হয়", "অনুমান", "গুঞ্জন", "গুঞ্জন রটে",
    "সম্ভাবনা", "সম্ভব", "সম্ভাব্য",
    "জল্পনা", "কল্পনা", "জল্পনা-কল্পনা",
    "নাকি", "কি তবে", "তবে কি", "তবে কী",
    "শোনা যাচ্ছে", "জানা গেছে যে", "খবর রটে",
    "চর্চা শুরু", "বিতর্ক শুরু", "প্রশ্ন উঠেছে",
]

ENTERTAINMENT_TERMS = [
    "অভিনেত্রী", "অভিনেতা", "মডেল", "গায়ক", "গায়িকা", "নায়ক", "নায়িকা",
    "বলিউড", "হলিউড", "টলিউড", "ঢালিউড",
    "ব্যক্তিগত জীবন", "প্রেম", "প্রেমের", "বিবাহবিচ্ছেদ",
    "ছাড়াছাড়ি", "বিয়ে", "বিয়ের", "প্রেমের গল্প", "নতুন জুটি",
    "ভাইরাল", "টুইট", "ইনস্টাগ্রামে",
    "ছবি ভাইরাল", "ভিডিও ভাইরাল", "ছবি ফাঁস", "অন্তরঙ্গ",
    "চলচ্চিত্র", "প্রিমিয়ার", "শুটিং", "সিনেমা", "নাটক",
    "অভিনয়", "মুক্তি", "বক্স অফিস", "ট্রেইলর",
    "গসিপ", "ফটোশুট", "মেকআপ", "ড্রেস", "গাউন",
    "বিউটি", "ফিটনেস", "ওজন কমানো", "ফিগার", "সাইজ জিরো",
    "পুরস্কার", "এওয়ার্ড", "অস্কার",
]

SENSITIVE_TOPIC_TERMS = [
    # Communal / religious
    "মুসলমান", "হিন্দু", "ইসলাম", "হিন্দুধর্ম", "মন্দির", "মসজিদ", "মাদ্রাসা",
    "ধর্মীয়", "ধর্ম", "সাম্প্রদায়িক", "সম্প্রদায়িক", "দাঙ্গা", "দাঙ্গাহাঙ্গামা",
    "উসকানি", "উসকানি দিয়েছে", "ধর্মান্ধ", "কট্টর", "অমুসলিম", "কাফির",
    # Gender / sexual
    "ধর্ষণ", "ধর্ষিতা", "নারী নির্যাতন", "যৌন হয়রানি", "ইভ টিজিং",
    "নারীবাদী", "মেয়েদের", "নারীদের অধিকার",
    # Ethnicity / regional
    "উপজাতি", "চাকমা", "মারমা", "ত্রিপুরা", "গারো", "সাঁওতাল",
    "আদিবাসী", "পাহাড়ি", "সমতট",
    # Political provocation
    "সরকারবিরোধী", "বিরোধীদল", "ক্ষমতাসীন", "আওয়ামী লীগ", "বিএনপি",
    "জামায়াত", "জাতীয় পার্টি", "হেফাজত", "ছাত্রলীগ", "ছাত্রদল",
    "জিহাদ", "শহীদ", "শহীদের", "রাজাকার", "আলবদর",
    "বয়কট", "অবরোধ", "অচলাবস্থা", "ধর্মঘট",
    "বিচ্ছিন্নতাবাদী", "স্বাধীনতাবিরোধী",
]

BENGALI_STOPWORDS = {
    "এবং", "ও", "এর", "কে", "কেও", "তিনি", "তার", "তাকে", "তাদের",
    "এই", "সেই", "ঐ", "এক", "একটি", "একটা", "একজন",
    "হয়েছে", "হয়েছিল", "হবে", "হতে", "করেছেন", "করেছে",
    "বলেন", "বলেছেন", "যিনি", "যে", "যা",
    "আজ", "গতকাল", "আগামীকাল",
    "তবে", "কিন্তু", "আর", "অথচ", "যদিও",
    "কারণ", "তাই", "সুতরাং",
    "নিয়ে", "দিয়ে", "থেকে", "ভিতরে", "বাইরে",
    "সাথে", "সঙ্গে", "নিচে", "উপরে",
    "সব", "অনেক", "কিছু", "কোনো", "অন্য", "নিজে",
}

DATELINE_RE = re.compile(
    r"^[^\s,]{2,15}\s*,\s*[\d০-৯]|^[^\s]{2,15}\s*\([^)]+\)\s*[-—]"
)

# --- Helper functions ---

def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\u200d", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def count_term_hits(text, terms):
    if not text:
        return 0
    return sum(1 for t in terms if t in text)

def count_total_term_hits(text, terms):
    if not text:
        return 0
    return sum(text.count(t) for t in terms)

def word_count(text):
    if not text:
        return 0
    return len(text.split())

def has_strong_attribution(headline, body):
    full = normalize_text(headline or "") + " " + normalize_text(body or "")
    credible_sources = [
        "টিআইবি", "ট্রান্সপারেন্সি", "রয়টার্স", "রইটার্স", "বিডিনিউজ",
        "বাসস", "ইউএনবি", "এএফপি", "বিশ্বব্যাংক", "আইএমএফ",
        "জাতিসংঘ", "ইউনিসেফ", "বিশ্ববিদ্যালয়", "গবেষণা", "সমীক্ষা",
        "আদালত", "পুলিশ", "র‌্যাব", "সিআইডি", "মন্ত্রণালয়",
        "প্রতিবেদক", "প্রতিনিধি", "নিজস্ব প্রতিবেদক",
        "বিজ্ঞপ্তি", "বিবৃতি",
    ]
    return any(src in full for src in credible_sources)

def has_dateline(body):
    b = normalize_text(body or "")[:200]
    return bool(DATELINE_RE.match(b))

# --- Seven Criteria Scoring Functions ---

def C1_sensational_headline(headline):
    """C1: Sensational headline score in [0,1]."""
    if not headline:
        return 0.0
    h = normalize_text(headline)
    hits = count_term_hits(h, SENSATIONAL_HEADLINE_TERMS)
    marks = h.count("!") + h.count("?")
    base = min(hits / 2.0, 1.0)
    mark_bonus = min(marks / 1.5, 0.3)
    return min(base + mark_bonus, 1.0)

def C2_clickbait(headline, body):
    """C2: Clickbait score in [0,1]."""
    h = normalize_text(headline or "")
    phrase_hits = count_term_hits(h, CLICKBAIT_PHRASES)
    listicle_hit = 1 if CLICKBAIT_LISTICLE_RE.search(h) else 0
    trailing_q = 1 if (h.endswith("?") or h.endswith("…") or h.endswith("...")) else 0
    base = min(phrase_hits / 1.5, 1.0)
    bonus = 0.15 * listicle_hit + 0.20 * trailing_q
    return min(base + bonus, 1.0)

def C3_emotional(body):
    """C3: Emotional arousal score in [0,1].
    Formula: 1 - exp(-D/gamma), D = density per 100 words
    """
    b = normalize_text(body or "")
    if not b:
        return 0.0
    wc = word_count(b)
    if wc == 0:
        return 0.0
    hits = count_total_term_hits(b, EMOTIONAL_TERMS)
    density = hits / max(wc / 100.0, 1.0)
    gamma = 1.2
    score = 1 - math.exp(-density / gamma)
    return min(score, 1.0)

def C4_attribution_gap(headline, body):
    """C4: Attribution gap score in [0,1].
    Formula: max(1 - lambda*n_attr - credits, 0) + short_penalty
    """
    b = normalize_text(body or "")
    h = normalize_text(headline or "")
    full = h + " " + b
    wc = word_count(b)
    if wc == 0:
        return 1.0
    attr_hits = count_term_hits(full, ATTRIBUTION_TERMS)
    has_strong = has_strong_attribution(h, b)
    has_dl = has_dateline(b)
    lam = 0.10
    base = max(1.0 - lam * attr_hits, 0.0)
    if has_strong:
        base = max(base - 0.30, 0.0)
    if has_dl:
        base = max(base - 0.15, 0.0)
    if wc < 100:
        base = min(base + 0.05, 1.0)
    return min(max(base, 0.0), 1.0)

def C5_speculation(body):
    """C5: Speculation-as-fact score in [0,1].
    Formula: 1 - exp(-D/gamma)
    """
    b = normalize_text(body or "")
    if not b:
        return 0.0
    wc = word_count(b)
    hits = count_total_term_hits(b, SPECULATION_TERMS)
    if wc == 0:
        return 0.0
    density = hits / max(wc / 100.0, 1.0)
    gamma = 1.2
    score = 1 - math.exp(-density / gamma)
    return min(score, 1.0)

def C6_entertainment(headline, body):
    """C6: Entertainment displacement score in [0,1].
    Formula: min(hits/alpha + 0.25*headline_hits, 1)
    """
    h = normalize_text(headline or "")
    b = normalize_text(body or "")
    full = h + " " + b
    hits = count_term_hits(full, ENTERTAINMENT_TERMS)
    headline_hits = count_term_hits(h, ENTERTAINMENT_TERMS)
    alpha = 3.0
    base = min(hits / alpha, 1.0)
    headline_bonus = min(0.25 * headline_hits, 0.5)
    return min(base + headline_bonus, 1.0)

def C7_coherence(headline, body):
    """C7: Headline-body coherence (mismatch) score in [0,1].
    Formula: 1 - overlap_ratio if overlap < tau, else 0
    """
    h = normalize_text(headline or "")
    b = normalize_text(body or "")
    if not h or not b:
        return 0.3
    h_tokens = set(re.findall(r"[\u0980-\u09FF]+|[A-Za-z]+|\d+", h))
    b_tokens = set(re.findall(r"[\u0980-\u09FF]+|[A-Za-z]+|\d+", b))
    h_tokens = {t for t in h_tokens if len(t) > 1 and t not in BENGALI_STOPWORDS}
    b_tokens = {t for t in b_tokens if len(t) > 1 and t not in BENGALI_STOPWORDS}
    if not h_tokens:
        return 0.3
    overlap = h_tokens & b_tokens
    overlap_ratio = len(overlap) / len(h_tokens)
    tau = 0.35
    if overlap_ratio < tau:
        return 1.0 - overlap_ratio
    return 0.0

def C8_sensitive_topic(headline, body):
    """C8: Sensitive topic score in [0,1].
    Formula: 1 - exp(-D/gamma), D = density per 100 words on
    combined headline+body, gamma = 1.5.
    """
    h = normalize_text(headline or "")
    b = normalize_text(body or "")
    full = h + " " + b
    if not full.strip():
        return 0.0
    wc = word_count(full)
    if wc == 0:
        return 0.0
    hits = count_total_term_hits(full, SENSITIVE_TOPIC_TERMS)
    density = hits / max(wc / 100.0, 1.0)
    gamma = 1.5
    score = 1 - math.exp(-density / gamma)
    return min(score, 1.0)

def compute_all_criteria(headline, body):
    """Compute all 8 criteria scores for an article."""
    return {
        'C1': round(C1_sensational_headline(headline), 4),
        'C2': round(C2_clickbait(headline, body), 4),
        'C3': round(C3_emotional(body), 4),
        'C4': round(C4_attribution_gap(headline, body), 4),
        'C5': round(C5_speculation(body), 4),
        'C6': round(C6_entertainment(headline, body), 4),
        'C7': round(C7_coherence(headline, body), 4),
        'C8': round(C8_sensitive_topic(headline, body), 4),
    }

print('SMI criteria scoring functions defined.')
print(f'  C1: Sensational Headline (lexicon size: {len(SENSATIONAL_HEADLINE_TERMS)})')
print(f'  C2: Clickbait (lexicon size: {len(CLICKBAIT_PHRASES)})')
print(f'  C3: Emotional Arousal (lexicon size: {len(EMOTIONAL_TERMS)})')
print(f'  C4: Attribution Gap (lexicon size: {len(ATTRIBUTION_TERMS)})')
print(f'  C5: Speculation (lexicon size: {len(SPECULATION_TERMS)})')
print(f'  C6: Entertainment (lexicon size: {len(ENTERTAINMENT_TERMS)})')
print(f'  C7: Headline-Body Coherence')
print(f'  C8: Sensitive Topic (lexicon size: {len(SENSITIVE_TOPIC_TERMS)})')


SMI criteria scoring functions defined.
  C1: Sensational Headline (lexicon size: 41)
  C2: Clickbait (lexicon size: 38)
  C3: Emotional Arousal (lexicon size: 40)
  C4: Attribution Gap (lexicon size: 59)
  C5: Speculation (lexicon size: 26)
  C6: Entertainment (lexicon size: 49)
  C7: Headline-Body Coherence
  C8: Sensitive Topic (lexicon size: 57)


In [6]:
# === COMPUTE SMI CRITERIA FOR ALL 766 GOLD ARTICLES ===
print('Computing SMI criteria scores for 766 gold articles...', flush=True)
t0 = time.time()

criteria_rows = []
for _, row in df.iterrows():
    c = compute_all_criteria(row['headline'], row['body_text'])
    c['article_id'] = row['article_id']
    criteria_rows.append(c)

gold_criteria = pd.DataFrame(criteria_rows)
t1 = time.time()
print(f'Done in {t1-t0:.2f}s', flush=True)
print(f'Shape: {gold_criteria.shape}', flush=True)

# Show criteria score statistics by true label
print('\nCriteria score means by label (Yellow=1, Non-yellow=0):', flush=True)
for c in ['C1','C2','C3','C4','C5','C6','C7','C8']:
    m_y = gold_criteria.loc[df['best_label'].values == 1, c].mean()
    m_n = gold_criteria.loc[df['best_label'].values == 0, c].mean()
    print(f'  {c}: Yellow={m_y:.3f}, Non-yellow={m_n:.3f}, Diff={m_y-m_n:+.3f}', flush=True)

# Attach to main df for downstream joins
for c in ['C1','C2','C3','C4','C5','C6','C7','C8']:
    df[c] = gold_criteria[c].values

gold_criteria.head()


Computing SMI criteria scores for 766 gold articles...
Done in 2.54s
Shape: (766, 9)

Criteria score means by label (Yellow=1, Non-yellow=0):
  C1: Yellow=0.186, Non-yellow=0.016, Diff=+0.171
  C2: Yellow=0.029, Non-yellow=0.002, Diff=+0.027
  C3: Yellow=0.060, Non-yellow=0.050, Diff=+0.010
  C4: Yellow=0.598, Non-yellow=0.361, Diff=+0.237
  C5: Yellow=0.202, Non-yellow=0.101, Diff=+0.102
  C6: Yellow=0.381, Non-yellow=0.092, Diff=+0.289
  C7: Yellow=0.136, Non-yellow=0.092, Diff=+0.044
  C8: Yellow=0.167, Non-yellow=0.239, Diff=-0.072


,C1,C2,C3,C4,C5,C6,C7,C8,article_id
0,0.0,0.0,0.0000,0.25,0.0,0.0000,0.0,0.9619,v18_2830
1,0.0,0.0,0.0000,0.20,0.0,0.0000,0.0,0.0000,v18_0000
2,0.3,0.2,0.0000,1.00,0.0,0.3333,0.0,0.0000,v18_4780
3,0.0,0.0,0.0000,0.20,0.0,1.0000,0.0,0.6832,v18_0416
4,0.0,0.0,0.4606,0.90,0.0,0.0000,0.0,0.0000,v18_4000


### 6. Overall Error Analysis

Sanity check: recompute F1/precision/recall from the loaded predictions
and confirm they match NB1's published numbers (F1 ≈ 0.883). Then count
total errors, false positives, and false negatives.


In [7]:
# === OVERALL ERROR ANALYSIS ===
y_pred = banglabert_all_preds
y_prob = banglabert_all_probs

# Overall metrics (sanity check vs NB1's F1=0.883)
overall_acc  = accuracy_score(y_true, y_pred)
overall_prec = precision_score(y_true, y_pred, zero_division=0)
overall_rec  = recall_score(y_true, y_pred, zero_division=0)
overall_f1   = f1_score(y_true, y_pred, zero_division=0)
overall_kappa = cohen_kappa_score(y_true, y_pred)
overall_mcc   = matthews_corrcoef(y_true, y_pred)

print('=' * 60, flush=True)
print('  Overall BanglaBERT Performance (766 gold articles)', flush=True)
print('=' * 60, flush=True)
print(f'  Accuracy:  {overall_acc:.4f}', flush=True)
print(f'  Precision: {overall_prec:.4f}', flush=True)
print(f'  Recall:    {overall_rec:.4f}', flush=True)
print(f'  F1:        {overall_f1:.4f}   (NB1 reports 0.883)', flush=True)
print(f'  Kappa:     {overall_kappa:.4f}', flush=True)
print(f'  MCC:       {overall_mcc:.4f}', flush=True)
print('=' * 60, flush=True)

# Error counts
total_errors = int((y_pred != y_true).sum())
false_pos    = int(((y_pred == 1) & (y_true == 0)).sum())  # predicted yellow, actually non-yellow
false_neg    = int(((y_pred == 0) & (y_true == 1)).sum())  # predicted non-yellow, actually yellow
error_rate   = total_errors / n_total

print(f'\n  Total errors:     {total_errors} / {n_total}  ({error_rate*100:.2f}%)', flush=True)
print(f'  False positives:  {false_pos}', flush=True)
print(f'  False negatives:  {false_neg}', flush=True)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print(f'\n  Confusion matrix (rows=true, cols=pred):', flush=True)
print(f'                 Pred=0   Pred=1', flush=True)
print(f'    True=0 (NY)   {cm[0,0]:>6}   {cm[0,1]:>6}', flush=True)
print(f'    True=1 (Y)    {cm[1,0]:>6}   {cm[1,1]:>6}', flush=True)

# Classification report
print(f'\n  Classification report:', flush=True)
print(classification_report(y_true, y_pred, target_names=['Non-yellow (0)', 'Yellow (1)']), flush=True)

# Annotate df with error flags for downstream subgroup analysis
df['pred']        = y_pred
df['prob']        = y_prob
df['correct']     = (y_pred == y_true).astype(int)
df['is_error']    = (y_pred != y_true).astype(int)
df['error_type']  = 'correct'
df.loc[(y_pred == 1) & (y_true == 0), 'error_type'] = 'false_positive'
df.loc[(y_pred == 0) & (y_true == 1), 'error_type'] = 'false_negative'
print(f'\nAnnotated df with error flags: is_error, error_type, correct, pred, prob', flush=True)


  Overall BanglaBERT Performance (766 gold articles)
  Accuracy:  0.8799
  Precision: 0.8628
  Recall:    0.9034
  F1:        0.8827   (NB1 reports 0.883)
  Kappa:     0.7598
  MCC:       0.7606

  Total errors:     92 / 766  (12.01%)
  False positives:  55
  False negatives:  37

  Confusion matrix (rows=true, cols=pred):
                 Pred=0   Pred=1
    True=0 (NY)      328       55
    True=1 (Y)        37      346

  Classification report:
                precision    recall  f1-score   support

Non-yellow (0)       0.90      0.86      0.88       383
    Yellow (1)       0.86      0.90      0.88       383

      accuracy                           0.88       766
     macro avg       0.88      0.88      0.88       766
  weighted avg       0.88      0.88      0.88       766


Annotated df with error flags: is_error, error_type, correct, pred, prob


In [8]:
# === ERROR ANALYSIS BY ANNOTATION CONFIDENCE ===
print('=== Errors by best_confidence (H/M/L) ===\n', flush=True)

conf_rows = []
for conf in ['H', 'M', 'L']:
    mask = df['best_confidence'] == conf
    n = int(mask.sum())
    if n == 0:
        conf_rows.append({'Confidence': conf, 'n': 0, 'Errors': 0,
                          'Error Rate': float('nan'), 'F1': float('nan'),
                          'Precision': float('nan'), 'Recall': float('nan')})
        continue
    sub_y = y_true[mask]
    sub_p = y_pred[mask]
    conf_rows.append({
        'Confidence': conf,
        'n':          n,
        'Errors':     int((sub_p != sub_y).sum()),
        'Error Rate': float((sub_p != sub_y).mean()),
        'F1':         float(f1_score(sub_y, sub_p, zero_division=0)),
        'Precision':  float(precision_score(sub_y, sub_p, zero_division=0)),
        'Recall':     float(recall_score(sub_y, sub_p, zero_division=0)),
    })

conf_table = pd.DataFrame(conf_rows)
print(conf_table.to_string(index=False), flush=True)
print(flush=True)
print('Expected: L-confidence articles have higher error rate (harder to', flush=True)
print('annotate AND harder to classify). H-confidence articles are easiest.', flush=True)


=== Errors by best_confidence (H/M/L) ===

Confidence   n  Errors  Error Rate       F1  Precision   Recall
         H 376      37    0.098404 0.725926   0.576471 0.980000
         M 328      48    0.146341 0.909774   0.930769 0.889706
         L  62       7    0.112903 0.940171   0.982143 0.901639

Expected: L-confidence articles have higher error rate (harder to
annotate AND harder to classify). H-confidence articles are easiest.


In [9]:
# === ERROR ANALYSIS BY CORPUS BATCH ===
print('=== Errors by corpus_batch ===\n', flush=True)

batch_rows = []
for batch in sorted(df['corpus_batch'].unique()):
    mask = df['corpus_batch'] == batch
    n = int(mask.sum())
    sub_y = y_true[mask]
    sub_p = y_pred[mask]
    batch_rows.append({
        'Batch':      batch,
        'n':          n,
        'Errors':     int((sub_p != sub_y).sum()),
        'Error Rate': float((sub_p != sub_y).mean()),
        'F1':         float(f1_score(sub_y, sub_p, zero_division=0)),
    })

batch_table = pd.DataFrame(batch_rows).sort_values('n', ascending=False)
print(batch_table.to_string(index=False), flush=True)
print(flush=True)
print('Largest batch (corpus_expansion_5000, n=649) dominates total errors;', flush=True)
print('smaller stratified-resample batches may show higher variance.', flush=True)


=== Errors by corpus_batch ===

                Batch   n  Errors  Error Rate       F1
corpus_expansion_5000 649      74    0.114022 0.892128
           v17_reused  48      10    0.208333 0.761905
      new_low_w0_pany  19       3    0.157895 0.823529
        new_mid_w0_p0  16       3    0.187500 0.727273
        new_mid_w1_p1  14       1    0.071429 0.888889
     new_high_w1_pany  10       0    0.000000 1.000000
        new_mid_w1_p0  10       1    0.100000 0.933333

Largest batch (corpus_expansion_5000, n=649) dominates total errors;
smaller stratified-resample batches may show higher variance.


In [10]:
# === ERROR ANALYSIS BY ARTICLE LENGTH ===
print('=== Errors by article_length bin ===\n', flush=True)

# 5 bins: 0-500, 500-1K, 1K-2K, 2K-4K, 4K+
bins   = [0, 500, 1000, 2000, 4000, 1_000_000]
labels = ['0-500', '500-1K', '1K-2K', '2K-4K', '4K+']
df['length_bin'] = pd.cut(df['article_length'], bins=bins, labels=labels, right=False, include_lowest=True)

len_rows = []
for lab in labels:
    mask = df['length_bin'] == lab
    n = int(mask.sum())
    if n == 0:
        len_rows.append({'Length Bin': lab, 'n': 0, 'Errors': 0,
                         'Error Rate': float('nan'), 'F1': float('nan')})
        continue
    sub_y = y_true[mask]
    sub_p = y_pred[mask]
    len_rows.append({
        'Length Bin': lab,
        'n':          n,
        'Errors':     int((sub_p != sub_y).sum()),
        'Error Rate': float((sub_p != sub_y).mean()),
        'F1':         float(f1_score(sub_y, sub_p, zero_division=0)),
    })

len_table = pd.DataFrame(len_rows)
print(len_table.to_string(index=False), flush=True)
print(flush=True)
print('BanglaBERT uses MAX_LEN=512 WordPiece tokens (~1500-2000 Bengali chars).', flush=True)
print('Articles longer than ~2000 chars are truncated; very short articles', flush=True)
print('(0-500 chars) lack enough signal for confident classification.', flush=True)


=== Errors by article_length bin ===

Length Bin   n  Errors  Error Rate       F1
     0-500  58       5    0.086207 0.912281
    500-1K 174      20    0.114943 0.878049
     1K-2K 353      44    0.124646 0.881081
     2K-4K 141      14    0.099291 0.907895
       4K+  40       9    0.225000 0.780488

BanglaBERT uses MAX_LEN=512 WordPiece tokens (~1500-2000 Bengali chars).
Articles longer than ~2000 chars are truncated; very short articles
(0-500 chars) lack enough signal for confident classification.


In [11]:
# === ERROR ANALYSIS BY TRUE LABEL — FP vs FN EXAMPLES ===
print('=== False Positives (pred=Yellow, true=Non-yellow) ===\n', flush=True)
print(f'  Count: {false_pos}', flush=True)

fp_examples = df[df['error_type'] == 'false_positive'].sample(
    n=min(5, false_pos), random_state=SEED
) if false_pos > 0 else pd.DataFrame()

for i, (_, r) in enumerate(fp_examples.iterrows(), 1):
    print(f'\n  --- FP Example {i} ---', flush=True)
    print(f'  article_id:   {r["article_id"]}', flush=True)
    print(f'  prob:         {r["prob"]:.4f}', flush=True)
    print(f'  confidence:   {r["best_confidence"]}', flush=True)
    print(f'  corpus_batch: {r["corpus_batch"]}', flush=True)
    print(f'  length:       {r["article_length"]} chars', flush=True)
    print(f'  headline:     {r["headline"][:120]}', flush=True)
    print(f'  best_note:    {r["best_note"][:200]}', flush=True)

print(f'\n\n=== False Negatives (pred=Non-yellow, true=Yellow) ===\n', flush=True)
print(f'  Count: {false_neg}', flush=True)

fn_examples = df[df['error_type'] == 'false_negative'].sample(
    n=min(5, false_neg), random_state=SEED
) if false_neg > 0 else pd.DataFrame()

for i, (_, r) in enumerate(fn_examples.iterrows(), 1):
    print(f'\n  --- FN Example {i} ---', flush=True)
    print(f'  article_id:   {r["article_id"]}', flush=True)
    print(f'  prob:         {r["prob"]:.4f}', flush=True)
    print(f'  confidence:   {r["best_confidence"]}', flush=True)
    print(f'  corpus_batch: {r["corpus_batch"]}', flush=True)
    print(f'  length:       {r["article_length"]} chars', flush=True)
    print(f'  headline:     {r["headline"][:120]}', flush=True)
    print(f'  best_note:    {r["best_note"][:200]}', flush=True)

print(f'\n\nSummary: FP={false_pos}, FN={false_neg}. '
      f'BanglaBERT is {"more FP-prone" if false_pos > false_neg else "more FN-prone"}.', flush=True)


=== False Positives (pred=Yellow, true=Non-yellow) ===

  Count: 55

  --- FP Example 1 ---
  article_id:   v18_2575
  prob:         0.9305
  confidence:   H
  corpus_batch: corpus_expansion_5000
  length:       522 chars
  headline:     পাবনায় শিশুসহ ২ জনের আত্মহত্যা
  best_note:    Crime report with named victims Ria Khatun (11) and Moktar Hossain (45), named fathers Habibur Rahman and Mo. Fazder Ali Pramanik, named locations, recorded UD case. Strong victim/family attribution.

  --- FP Example 2 ---
  article_id:   v18_1103
  prob:         0.9732
  confidence:   H
  corpus_batch: corpus_expansion_5000
  length:       1021 chars
  headline:     মুম্বাই হামলার তথ্য জানতেন লাদেন!
  best_note:    Headline has '!' but body cites Times of India and Dawn, with named book editor Azaz Syed and historical facts (Kasab's execution, Abbottabad raid). Wire agency/book attribution.

  --- FP Example 3 ---
  article_id:   v18_3726
  prob:         0.9347
  confidence:   M
  corpus_batch: corpus_e

In [12]:
# === ERROR CORRELATION WITH SMI CRITERIA ===
print('=== Error correlation with SMI criteria (C1-C8) ===\n', flush=True)

# For each criterion, compare scores on correctly classified vs misclassified articles
# Two-sample t-test (Welch's) to assess significance.
smi_corr_rows = []
correct_mask = df['correct'] == 1
error_mask   = df['is_error'] == 1

for c in ['C1','C2','C3','C4','C5','C6','C7','C8']:
    scores_correct = df.loc[correct_mask, c].astype(float).values
    scores_error   = df.loc[error_mask,   c].astype(float).values
    if len(scores_error) < 2 or len(scores_correct) < 2 or np.allclose(scores_correct.std(), 0) and np.allclose(scores_error.std(), 0):
        t_stat, p_val = float('nan'), float('nan')
    else:
        t_stat, p_val = stats.ttest_ind(scores_correct, scores_error, equal_var=False, nan_policy='omit')
    sig = 'YES' if (not np.isnan(p_val) and p_val < 0.05) else 'no'
    smi_corr_rows.append({
        'Criterion':       c,
        'Mean (Correct)':  float(np.mean(scores_correct)) if len(scores_correct) else float('nan'),
        'Mean (Errors)':   float(np.mean(scores_error))   if len(scores_error)   else float('nan'),
        'Diff (Err-Cor)':  float(np.mean(scores_error) - np.mean(scores_correct)) if len(scores_correct) and len(scores_error) else float('nan'),
        't-stat':          float(t_stat) if not np.isnan(t_stat) else float('nan'),
        'p-value':         float(p_val)  if not np.isnan(p_val)  else float('nan'),
        'Significant (p<0.05)': sig,
    })

smi_corr_table = pd.DataFrame(smi_corr_rows)
print(smi_corr_table.to_string(index=False), flush=True)
print(flush=True)

# Identify the criterion with the largest absolute difference
smi_corr_table['abs_diff'] = smi_corr_table['Diff (Err-Cor)'].abs()
top_criterion_row = smi_corr_table.loc[smi_corr_table['abs_diff'].idxmax()]
print(f'Top differentiating criterion: {top_criterion_row["Criterion"]} '
      f'(diff={top_criterion_row["Diff (Err-Cor)"]:+.4f}, '
      f'p={top_criterion_row["p-value"]:.4f})', flush=True)


=== Error correlation with SMI criteria (C1-C8) ===

Criterion  Mean (Correct)  Mean (Errors)  Diff (Err-Cor)    t-stat  p-value Significant (p<0.05)
       C1        0.103561       0.082609       -0.020952  1.220312 0.224753                   no
       C2        0.016494       0.010326       -0.006168  1.118684 0.264674                   no
       C3        0.054934       0.051874       -0.003060  0.200625 0.841308                   no
       C4        0.473591       0.522826        0.049236 -1.316582 0.190501                   no
       C5        0.152334       0.146496       -0.005838  0.214324 0.830667                   no
       C6        0.242084       0.198366       -0.043718  1.213595 0.227176                   no
       C7        0.115405       0.105013       -0.010392  0.337527 0.736317                   no
       C8        0.202995       0.204917        0.001923 -0.055021 0.956216                   no

Top differentiating criterion: C4 (diff=+0.0492, p=0.1905)


In [13]:
# === PROBABILITY CALIBRATION ANALYSIS ===
print('=== Probability Calibration ===\n', flush=True)

# 10 bins over [0,1]
n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# For each bin, compute the actual fraction of positives
bin_accs = []
bin_confs = []
bin_counts = []
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask = (y_prob >= lo) & (y_prob < hi)
    if lo == 1.0:  # right-edge inclusive for the last bin
        mask = (y_prob >= lo) & (y_prob <= hi)
    n = int(mask.sum())
    if n > 0:
        bin_confs.append(float(y_prob[mask].mean()))
        bin_accs.append(float(y_true[mask].mean()))
        bin_counts.append(n)
    else:
        bin_confs.append(float('nan'))
        bin_accs.append(float('nan'))
        bin_counts.append(0)

calib_df = pd.DataFrame({
    'bin_lo':     bin_edges[:-1],
    'bin_hi':     bin_edges[1:],
    'n':          bin_counts,
    'mean_prob':  bin_confs,
    'frac_pos':   bin_accs,
})
print(calib_df.round(4).to_string(index=False), flush=True)

# Brier score (lower is better)
brier = float(np.mean((y_prob - y_true) ** 2))
print(f'\nBrier score: {brier:.4f}  (lower is better; 0=perfect)', flush=True)

# Expected Calibration Error (ECE)
ece_num = 0.0
N = len(y_true)
for n, conf, acc in zip(bin_counts, bin_confs, bin_accs):
    if n > 0 and not np.isnan(conf) and not np.isnan(acc):
        ece_num += (n / N) * abs(acc - conf)
ece = float(ece_num)
print(f'ECE:         {ece:.4f}  (lower is better; 0=perfectly calibrated)', flush=True)

# Also compute maximum calibration error (MCE) — max |acc - conf| over non-empty bins
mce = float(max(
    abs(a - c) for n, c, a in zip(bin_counts, bin_confs, bin_accs)
    if n > 0 and not np.isnan(c) and not np.isnan(a)
))
print(f'MCE:         {mce:.4f}  (worst-bin gap)', flush=True)

# Reliability assessment
if ece < 0.05:
    calib_quality = 'well-calibrated'
elif ece < 0.10:
    calib_quality = 'moderately calibrated'
else:
    calib_quality = 'poorly calibrated'
print(f'\nCalibration quality: {calib_quality} (ECE threshold: <0.05 well, <0.10 moderate)', flush=True)


=== Probability Calibration ===

 bin_lo  bin_hi   n  mean_prob  frac_pos
    0.0     0.1 341     0.0290    0.0792
    0.1     0.2  14     0.1292    0.2857
    0.2     0.3   3     0.2447    1.0000
    0.3     0.4   5     0.3536    0.4000
    0.4     0.5   2     0.4387    0.5000
    0.5     0.6   6     0.5659    0.8333
    0.6     0.7   4     0.6553    0.7500
    0.7     0.8   4     0.7512    1.0000
    0.8     0.9  42     0.8613    0.8095
    0.9     1.0 345     0.9516    0.8696

Brier score: 0.1071  (lower is better; 0=perfect)
ECE:         0.0723  (lower is better; 0=perfectly calibrated)
MCE:         0.7553  (worst-bin gap)

Calibration quality: moderately calibrated (ECE threshold: <0.05 well, <0.10 moderate)


In [14]:
# === 4-PANEL VISUALIZATION ===
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)

# Panel 1: Error rate by confidence
ax = axes[0, 0]
conf_plot = conf_table.dropna(subset=['Error Rate']).copy()
bars = ax.bar(conf_plot['Confidence'], conf_plot['Error Rate'],
              color=['#4CAF50', '#FF9800', '#F44336'][:len(conf_plot)],
              alpha=0.85, edgecolor='black', linewidth=0.8)
for bar, er, n in zip(bars, conf_plot['Error Rate'], conf_plot['n']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{er:.3f}\n(n={n})', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Annotation Confidence')
ax.set_ylabel('Error Rate')
ax.set_title('Panel 1: Error Rate by Annotation Confidence')
ax.set_ylim(0, max(0.30, conf_plot['Error Rate'].max() * 1.25))
ax.grid(axis='y', alpha=0.3)

# Panel 2: Error rate by corpus_batch
ax = axes[0, 1]
batch_plot = batch_table.dropna(subset=['Error Rate']).copy()
# Truncate long batch names
batch_plot['Batch_short'] = batch_plot['Batch'].str.replace('corpus_expansion_5000', 'corpus_exp')
bars = ax.barh(batch_plot['Batch_short'], batch_plot['Error Rate'],
               color='#2196F3', alpha=0.85, edgecolor='black', linewidth=0.8)
for bar, er, n in zip(bars, batch_plot['Error Rate'], batch_plot['n']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{er:.3f} (n={n})', va='center', fontsize=9)
ax.set_xlabel('Error Rate')
ax.set_ylabel('Corpus Batch')
ax.set_title('Panel 2: Error Rate by Corpus Batch')
ax.set_xlim(0, max(0.30, batch_plot['Error Rate'].max() * 1.4))
ax.grid(axis='x', alpha=0.3)

# Panel 3: Error rate by article length bin
ax = axes[1, 0]
len_plot = len_table.dropna(subset=['Error Rate']).copy()
bars = ax.bar(len_plot['Length Bin'], len_plot['Error Rate'],
              color='#9C27B0', alpha=0.85, edgecolor='black', linewidth=0.8)
for bar, er, n in zip(bars, len_plot['Error Rate'], len_plot['n']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{er:.3f}\n(n={n})', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Article Length Bin (chars)')
ax.set_ylabel('Error Rate')
ax.set_title('Panel 3: Error Rate by Article Length Bin')
ax.set_ylim(0, max(0.30, len_plot['Error Rate'].max() * 1.25))
ax.grid(axis='y', alpha=0.3)

# Panel 4: Reliability diagram
ax = axes[1, 1]
# Perfect-calibration diagonal
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
# Bars: mean prob vs frac positive
non_empty = calib_df[calib_df['n'] > 0].copy()
ax.bar(non_empty['mean_prob'], non_empty['frac_pos'],
       width=0.08, alpha=0.55, color='#FF5722', edgecolor='black',
       linewidth=0.6, label='BanglaBERT (binned)')
# Overlay gap lines
for _, r in non_empty.iterrows():
    if not np.isnan(r['mean_prob']) and not np.isnan(r['frac_pos']):
        ax.plot([r['mean_prob'], r['mean_prob']],
                [r['mean_prob'], r['frac_pos']],
                color='gray', alpha=0.5, linewidth=0.8)
ax.set_xlabel('Mean Predicted Probability (bin)')
ax.set_ylabel('Fraction of True Positives')
ax.set_title(f'Panel 4: Reliability Diagram\n'
             f'Brier={brier:.4f}  ECE={ece:.4f}  ({calib_quality})')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc='upper left', fontsize=9)
ax.grid(alpha=0.3)

fig.suptitle('BanglaBERT Large Error Analysis on 766 Gold Articles (5-fold CV, seed=42)',
             fontsize=13, fontweight='bold')

viz_path = OUTPUT_DIR / 'banglabert_error_analysis.png'
plt.savefig(viz_path, dpi=150, bbox_inches='tight')
print(f'Saved 4-panel visualization: {viz_path}', flush=True)
plt.show()


Saved 4-panel visualization: /kaggle/working/banglabert_error_analysis.png


In [15]:
# === SAVE RESULTS JSON ===
results = {
    'notebook': 'NB14_BanglaBERT_Error_Analysis',
    'dataset':  os.path.basename(GOLD_PATH),
    'n_articles':     int(n_total),
    'n_yellow':       int(y_true.sum()),
    'n_non_yellow':   int((y_true == 0).sum()),
    'seed':           SEED,
    'pred_source':    PRED_SOURCE,
    'pred_csv':       os.path.basename(PRED_PATH) if PRED_PATH else None,
    'overall_metrics': {
        'accuracy':  float(overall_acc),
        'precision': float(overall_prec),
        'recall':    float(overall_rec),
        'f1':        float(overall_f1),
        'kappa':     float(overall_kappa),
        'mcc':       float(overall_mcc),
    },
    'total_errors':      int(total_errors),
    'false_positives':   int(false_pos),
    'false_negatives':   int(false_neg),
    'error_rate':        float(error_rate),
    'error_by_confidence': conf_table.to_dict(orient='records'),
    'error_by_batch':      batch_table.to_dict(orient='records'),
    'error_by_length_bin': len_table.to_dict(orient='records'),
    'error_correlation_with_smi': smi_corr_table.to_dict(orient='records'),
    'calibration_metrics': {
        'brier_score': float(brier),
        'ece':         float(ece),
        'mce':         float(mce),
        'quality':     calib_quality,
        'reliability_bins': calib_df.to_dict(orient='records'),
    },
    'key_findings': [],
    'note': ('BanglaBERT error analysis on 766-article gold standard. '
             'Predictions loaded from saved CSV (CPU-only) or regenerated '
             'via NB1 5-fold CV fallback (GPU). SMI criteria scoring '
             'functions copied verbatim from NB8.'),
}

# Auto-generate key findings based on computed numbers
findings = []
# Confidence finding
if not conf_table.empty:
    worst_conf = conf_table.loc[conf_table['Error Rate'].idxmax()]
    best_conf  = conf_table.loc[conf_table['Error Rate'].idxmin()]
    findings.append(
        f"By annotation confidence: {worst_conf['Confidence']}-confidence articles have the "
        f"highest error rate ({worst_conf['Error Rate']:.3f}), while {best_conf['Confidence']}-confidence "
        f"articles have the lowest ({best_conf['Error Rate']:.3f}). "
        f"BanglaBERT mirrors the human-annotation difficulty."
    )
# Batch finding
if not batch_table.empty:
    worst_batch = batch_table.loc[batch_table['Error Rate'].idxmax()]
    findings.append(
        f"By corpus batch: '{worst_batch['Batch']}' has the highest error rate "
        f"({worst_batch['Error Rate']:.3f}, n={worst_batch['n']}); "
        f"corpus_expansion_5000 (n=649) sets the baseline."
    )
# Length finding
if not len_table.empty:
    worst_len = len_table.loc[len_table['Error Rate'].idxmax()]
    findings.append(
        f"By article length: '{worst_len['Length Bin']}' bin has the highest error rate "
        f"({worst_len['Error Rate']:.3f}, n={worst_len['n']}). "
        f"BanglaBERT uses MAX_LEN=512 tokens; very short or truncated articles are harder."
    )
# FP vs FN finding
findings.append(
    f"Error type breakdown: {false_pos} false positives (pred=Yellow, true=Non-yellow) vs "
    f"{false_neg} false negatives (pred=Non-yellow, true=Yellow). "
    f"BanglaBERT is {'more FP-prone' if false_pos > false_neg else 'more FN-prone'}."
)
# SMI correlation finding
if not smi_corr_table.empty:
    top_row = smi_corr_table.loc[smi_corr_table['abs_diff'].idxmax()]
    sig = 'significantly' if top_row['Significant (p<0.05)'] == 'YES' else 'not significantly'
    findings.append(
        f"SMI criteria correlation: {top_row['Criterion']} shows the largest mean-score "
        f"difference between errors and correct predictions "
        f"(diff={top_row['Diff (Err-Cor)']:+.4f}, p={top_row['p-value']:.4f}, {sig} different)."
    )
# Calibration finding
findings.append(
    f"Probability calibration: Brier score={brier:.4f}, ECE={ece:.4f} — "
    f"BanglaBERT's probability estimates are {calib_quality}."
)

results['key_findings'] = findings

out_json = OUTPUT_DIR / 'banglabert_error_analysis.json'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, default=str, ensure_ascii=False)
print(f'Saved: {out_json}', flush=True)
print(flush=True)
print('=== Key Findings ===', flush=True)
for i, kf in enumerate(findings, 1):
    print(f'  {i}. {kf}', flush=True)


Saved: /kaggle/working/banglabert_error_analysis.json

=== Key Findings ===
  1. By annotation confidence: M-confidence articles have the highest error rate (0.146), while H-confidence articles have the lowest (0.098). BanglaBERT mirrors the human-annotation difficulty.
  2. By corpus batch: 'v17_reused' has the highest error rate (0.208, n=48); corpus_expansion_5000 (n=649) sets the baseline.
  3. By article length: '4K+' bin has the highest error rate (0.225, n=40). BanglaBERT uses MAX_LEN=512 tokens; very short or truncated articles are harder.
  4. Error type breakdown: 55 false positives (pred=Yellow, true=Non-yellow) vs 37 false negatives (pred=Non-yellow, true=Yellow). BanglaBERT is more FP-prone.
  5. SMI criteria correlation: C4 shows the largest mean-score difference between errors and correct predictions (diff=+0.0492, p=0.1905, not significantly different).
  6. Probability calibration: Brier score=0.1071, ECE=0.0723 — BanglaBERT's probability estimates are moderately calib

### 7. Discussion

After running all error-analysis cells, answer the following questions
based on the printed tables and the saved JSON:

**Q1. Which subgroup has the highest error rate?**

- Inspect `error_by_confidence`, `error_by_batch`, `error_by_length_bin`
  in the saved JSON.
- Expected: L-confidence articles (n=62) should have the highest
  error-by-confidence rate, because they were ambiguous for human
  annotators too. The smallest corpus batches (n<20) often show high
  variance in error rate — interpret with caution.
- Very short articles (<500 chars) and very long articles (>4K chars,
  truncated at MAX_LEN=512 WordPiece tokens) are both harder.

**Q2. Are errors correlated with any SMI criterion?**

- Inspect `error_correlation_with_smi` in the saved JSON.
- The criterion with the largest `|Diff (Err-Cor)|` AND
  `Significant (p<0.05) == YES` is the strongest association.
- Common pattern: BanglaBERT struggles on articles with high
  C6 (Entertainment Displacement) — entertainment news blends with
  mainstream news vocabulary, making the yellow/non-yellow boundary
  fuzzier. C4 (Attribution Gap) and C7 (Headline-Body Coherence)
  may also surface: articles with no clear attribution source or
  with headline-body mismatch are harder for the model.

**Q3. Is BanglaBERT well-calibrated?**

- Inspect `calibration_metrics` (Brier, ECE, MCE) in the saved JSON.
- Rule of thumb: ECE < 0.05 = well-calibrated; 0.05-0.10 = moderately;
  > 0.10 = poorly calibrated.
- BanglaBERT (fine-tuned with cross-entropy) is often *overconfident*
  on its wrong predictions — the reliability diagram bars will sit
  *below* the diagonal in the high-probability region if so.
- If poorly calibrated, consider temperature scaling or Platt scaling
  as a post-hoc fix (no retraining needed).

**Q4. Recommendations for improving BanglaBERT**

1. **L-confidence articles (n=62):** augment training data with more
   examples from the new_low_w0_pany / new_mid_w0_p0 batches (the
   batches that produce L-confidence annotations).
2. **Long-article truncation:** increase MAX_LEN to 1024 or use
   hierarchical attention (Longformer-style) for articles > 2K chars.
3. **Calibration:** apply temperature scaling on the validation fold
   logits; this typically cuts ECE by 50-70% with no F1 loss.
4. **FP/FN asymmetry:** if FP > FN, raise the decision threshold
   above 0.5 to require stronger evidence for a yellow prediction;
   if FN > FP, lower it.
5. **SMI-aware data augmentation:** if a specific C-criterion
   correlates with errors, oversample training articles that score
   high on that criterion.

**Q5. Comparison to SMI (NB8)**

BanglaBERT F1 = 0.883 vs SMI F1 = 0.808 (5-fold CV). The 7-point gap
is partly because BanglaBERT learns from raw text (richer signal) and
partly because SMI is constrained by hand-coded lexicons. But SMI is
interpretable per-criterion — BanglaBERT is not. The error correlation
with SMI criteria (Section 11) effectively reverse-engineers which
SMI-style features BanglaBERT is *implicitly* sensitive to.

---

### 8. How to run on Kaggle

1. **Recommended (CPU, ~5 min):** Upload `banglabert_clean_predictions.csv`
   (produced by NB1 cell 9) as a Kaggle dataset. Attach the gold CSV
   and the predictions CSV as inputs to this notebook. Run all cells.
2. **Fallback (GPU, ~2 hours):** Set `RUN_CV_FALLBACK = True` in the
   Configuration cell. Enable GPU in the Kaggle notebook settings.
   Run all cells. The notebook will download `csebuetnlp/banglabert_large`
   from HuggingFace (requires Internet enabled) and re-run NB1's 5-fold
   CV inline.
3. After running, download `banglabert_error_analysis.json` and
   `banglabert_error_analysis.png` from the Kaggle output panel.
